# For Google Colab 

**Uncomment when you run this file on Google Colab**

In [ ]:
'''
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

!mkdir '/content/drive/MyDrive/finalTerm'
!mkdir '/content/drive/MyDrive/finalTerm/dataset'

!pip install kaggle

from google.colab import files
# kaggle.json : Kaggle API token : Personal key
    # You can download your kaggle.json file from Kaggle website
    # 1. Enter https://www.kaggle.com/settings/account
    # 2. API > 'Create New Token'
    # 3. Download kaggle.json file
# Upload your kaggle.json file
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Dataset download
%cd /content/drive/MyDrive/finalTerm/dataset
!kaggle competitions download -c postech25-csed-441-final-project

# Unzip the dataset
!unzip postech25-csed-441-final-project.zip
'''

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Dataset & DataLoader
This section is baseline.

You can change this section if you want.

In [2]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import PIL
import os

class TrainDataset(Dataset):
    def __init__(self, root_path, transform, data_aug=None):
        super(TrainDataset,self).__init__()
        self.augmentation = data_aug
        self.root_path = root_path
        self.transform = transform
        self.image_list = list()
        self.annotation = pd.read_csv(os.path.join(self.root_path, '../Train_64.csv'))

        self.image_list = np.array(self.annotation.values.tolist())[:, 0]
        self.labels = np.array(self.annotation.values.tolist())[:, 1]

    def __len__(self):
        return len(self.annotation)

    def __getitem__(self,index):
        img = PIL.Image.open(os.path.join(self.root_path,str('Train_64'),self.image_list[index]))
        if self.augmentation is not None:
            img = self.augmentation(img)
        img = self.transform(img)
        return img, int(self.labels[index])

class TestDataset(Dataset):
    def __init__(self, root_path, transform, data_aug=None):
        super(TestDataset,self).__init__()
        self.root_path = root_path
        self.transform = transform
        self.image_list = list()
        self.annotation = pd.read_csv(os.path.join(self.root_path, '../Test_64.csv'))

        self.image_list = np.array(self.annotation.values.tolist())[:, 0]


    def __len__(self):
        return len(self.annotation)
    
    def __getitem__(self,index):

        img = PIL.Image.open(os.path.join(self.root_path,str('Test_64'),self.image_list[index]))
        img = self.transform(img)
        return img

In [3]:
import torchvision.transforms as transforms


transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

train_dataset = TrainDataset(root_path = '../input/postech25-csed-441-final-project/Train_64/',
                       transform = transform)

test_dataset = TestDataset(root_path = '../input/postech25-csed-441-final-project/Test_64/',
                       transform = transform)


train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=256,
                                           shuffle=True,
                                           num_workers=0,
                                           drop_last = True
                                          )


test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=256,
                                          shuffle=False,
                                          num_workers=0
                                         )

# Your Awesome Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)

class DownResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=2),
            nn.BatchNorm2d(out_ch)
        )

    def forward(self, x):
        identity = self.skip(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer1 = DownResidualBlock(64, 128)
        self.layer2 = DownResidualBlock(128, 256)
        self.layer3 = DownResidualBlock(256, 512)

        self.conv5 = nn.Conv2d(512, 15, 1)
        self.bn5 = nn.BatchNorm2d(15)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.adaptive_avg_pool2d(x, 1).squeeze()
        return x


model = Net()

# Model parameter checking

In [5]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 21441871
Parameter usage : 21.441871%


# Model training

In [ ]:
import tqdm
model = model.cuda()

loss = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

'''
Please note that the maximum number of epochs allowed is 15. For instance, if you opt to train a single model, you can allocate all 15 epochs to it. 
However, in the case of employing multiple models, the number of epochs should be divided among them accordingly.
'''

epochs = 1 # 15 epoch fix

for epoch in range(epochs):
    for step, (x,y) in enumerate(tqdm.tqdm(train_loader)):
        x = torch.FloatTensor(x).cuda()
        y = torch.LongTensor(y).cuda()
        
        output = model(x)
        cost = loss(output,y)
        
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()
        
        predict = torch.argmax(output,dim=1)
        
        train_accuracy = torch.mean((predict == y).float())

    print(" epoch : {:4d} , cost : {:.2f} , train_accuracy : {:.4f}".format(epoch,cost,train_accuracy))

  6%|▋         | 11/175 [10:09<2:29:48, 54.81s/it]

# Submit
Do not edit the submission code below.

In [21]:
submit = pd.read_csv('../input/postech25-csed-441-final-project/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 4311855
Parameter usage : 4.311855%


100%|██████████| 30/30 [01:14<00:00,  2.49s/it]


# Submit in Google Colab
**Uncomment when you run this file on Google Colab**

In [ ]:
'''
# !kaggle competitions submit -c postech25-csed-441-final-project -f submission.csv -m "<Your commit message>"
!kaggle competitions submit -c postech25-csed-441-final-project -f submission.csv -m "MY Submission"
'''